# tqdm-postfix-metrics — ex2: manual tqdm with phase-labeled set_description across train+val

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tqdm-postfix-metrics`. Running the final beacon cell reports progress against the `Logging: tqdm postfix metrics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: tqdm postfix metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tqdm-postfix-metrics`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tqdm-postfix-metrics"
DD_SUBTOPIC = "Logging: tqdm postfix metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `tqdm(total=N)` + `pbar.update(n)` + `set_description` — quick refresher

Wrapping an iterable in `tqdm(...)` is the common case. Sometimes you don't have a single iterable — you have multiple phases of work (train batches, val batches, log flush) and want ONE bar showing total progress. That calls for the manual API:

```python
pbar = tqdm(total=total_units)
for phase, units in phases:
    pbar.set_description(f'Phase: {phase}')
    for _ in range(units):
        do_work()
        pbar.update(1)         # advance by 1 unit
        pbar.set_postfix(loss=...)
pbar.close()
```

**`set_description` sets the LEFT label** (replaces `desc=` you'd pass at construction). **`update(n)` advances the bar by `n` units** — you control how many units per logical step.

**Always `close()` a manual `tqdm`.** Wrapping-an-iterable closes automatically when the iterator exhausts; the manual API does not.

### Exercise 2 — manual tqdm with phase-labeled set_description across train+val

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `tqdm(total=N)` + `pbar.update(1)` + `pbar.set_description` to drive a single progress bar across multiple phases (train + val), with the bar's left label switching per phase.
> Keywords: tqdm, manual-update, set-description, phase-labels
> ```

**KCs targeted:** `tqdm-manual-update`, `tqdm-set-description-phase`

Implement `ex2_manual_tqdm_phases(train_losses, val_losses)`. A single tqdm bar that spans train + val with dynamic phase labels:

1. Construct `pbar = tqdm(total=len(train_losses) + len(val_losses))`.
2. **Train phase**: `pbar.set_description('train')`. Loop over `train_losses`; for each loss, call `pbar.set_postfix(loss=f'{loss:.3f}')` and `pbar.update(1)`.
3. **Val phase**: `pbar.set_description('val')`. Loop over `val_losses`; for each loss, call `pbar.set_postfix(loss=f'{loss:.3f}')` and `pbar.update(1)`.
4. Call `pbar.close()`.
5. Return a dict: `{'final_n': pbar.n, 'final_desc': <phase>}`. tqdm stores the current bar position in `pbar.n` and the current description in `pbar.desc` — note that newer tqdm versions append `': '` to the desc, so **`rstrip(': ')` the value** before returning it so the caller sees just the phase name.

**Why not two separate bars?** A single bar makes the train+val ratio visually obvious. ARENA's eval-during-train loops use this to surface val cost vs train cost.

The test uses `disable=True`-style introspection — it inspects `pbar.n`, `pbar.total`, and the call history; nothing prints to the terminal during testing.

In [ ]:
from tqdm import tqdm

def ex2_manual_tqdm_phases(train_losses: list, val_losses: list) -> dict:
    """Single manual tqdm bar across train + val phases."""
    raise NotImplementedError()


def _test_ex2():
    out = ex2_manual_tqdm_phases([0.9, 0.7, 0.5], [0.4, 0.35])
    assert isinstance(out, dict), f'must return a dict, got {type(out)}'
    assert set(out.keys()) == {'final_n', 'final_desc'}, f'keys: {out.keys()}'

    # All 5 units must have been advanced.
    assert out['final_n'] == 5, f'expected pbar.n=5, got {out["final_n"]}'

    # Last set_description call was 'val' (val is the second phase).
    assert out['final_desc'] == 'val', (
        f'last set_description should be "val", got {out["final_desc"]!r} — '
        f'did you forget to set_description before the val phase?'
    )

    # Empty val phase — bar stops at len(train), desc stays at last set value (train).
    out2 = ex2_manual_tqdm_phases([1.0, 0.8], [])
    assert out2['final_n'] == 2, f'empty val: expected n=2, got {out2["final_n"]}'
    # When val phase is empty, val's set_description still ran (loop entered the val branch unconditionally).
    # Either 'train' (if you guard) or 'val' (if you unconditionally set) is acceptable —
    # the prompt says unconditionally set, so assert 'val'.
    assert out2['final_desc'] == 'val', (
        f'final_desc should be "val" (prompt says always set it before the val loop), '
        f'got {out2["final_desc"]!r}'
    )

    # Empty everything — n=0, total=0.
    out3 = ex2_manual_tqdm_phases([], [])
    assert out3['final_n'] == 0
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
from tqdm import tqdm

def ex2_manual_tqdm_phases(train_losses: list, val_losses: list) -> dict:
    pbar = tqdm(total=len(train_losses) + len(val_losses))
    pbar.set_description('train')
    for loss in train_losses:
        pbar.set_postfix(loss=f'{loss:.3f}')
        pbar.update(1)
    pbar.set_description('val')
    for loss in val_losses:
        pbar.set_postfix(loss=f'{loss:.3f}')
        pbar.update(1)
    # tqdm stores desc with a trailing ': ' separator on some versions;
    # rstrip it so the caller-facing label is just the phase name.
    desc = pbar.desc.rstrip(': ') if pbar.desc else ''
    out = {'final_n': pbar.n, 'final_desc': desc}
    pbar.close()
    return out
```

**`pbar.desc` vs `pbar.set_description(...)`.** `set_description` is the setter (it also re-renders the bar). `pbar.desc` is the attribute it writes to — fine to read directly. Newer tqdm versions append `': '` automatically when you `set_description`, so we `rstrip(': ')` in the return path to give callers just the phase name.

**`update(1)` vs ` update(n)`.** Pass `n` whenever a logical step covers `n` units — e.g. `pbar.update(batch_size)` for an examples-seen bar. Here we treat each loss as one unit.

**Why manual instead of wrapping a chained iterator.** `itertools.chain(train_losses, val_losses)` works for the bar itself but loses the phase boundary — you can't `set_description` mid-iteration without an external counter. Manual is clearer.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()